In [1]:
import time
_quickstart_started = time.perf_counter()  # cell execution, not browser queue time
import contextlib, gzip, hashlib, importlib.util, json, os, platform, shutil, subprocess, sys, tempfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from urllib.request import Request, urlopen

COMMIT = "0087fd42320ddc5742db19a8efa538f5cd32b551"
HELPER_PATH = "logs/loader-2026-10-22/scripts/colab_prepare.py"
HELPER_SHA256 = "2621cd8e2a28ff1b3808da4dd967fb8bf1dfcb862475e42dc9bdea6a64b2c5bf"
HF_BASE = "https://huggingface.co/datasets/Yakiv-Contour/panvram-hprc558/resolve/main"
SUMS_Q16K_SHA256 = "c17e433c95183e51da66c04526bbd81205807ba7030623a36f1d8545e7afd392"
REF_URL = ("https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/009/914/755/"
           "GCA_009914755.4_T2T-CHM13v2.0/GCA_009914755.4_T2T-CHM13v2.0_genomic.fna.gz")
REF_BYTES = 3156259565
REF_SHA256 = "51ab7d3c2a375ecff53ee2a1be03e58d57cc1601c948a860c03a78d724f26e4c"
REF_MD5 = "cd1e52ce400c027ed0b7ab4b9d613f5a"
BASES_SHA256 = "a6e4a74516252f28ac61291c9e11eba9e9cd53a576e014b3dfd51e4816333080"
work = Path(tempfile.mkdtemp(prefix="panvram-quickstart-", dir="/content"))
log_path = work / "quickstart.log"
receipt_path = work / "receipt.json"


class Tee:
    def __init__(self, screen, log):
        self.screen, self.log = screen, log
    def write(self, text):
        self.screen.write(text)
        self.log.write(text)
        self.log.flush()
        return len(text)
    def flush(self):
        self.screen.flush()
        self.log.flush()


def run(command, env=None):
    # Stream compiler output into the receipt log; never print the environment.
    with subprocess.Popen(command, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise RuntimeError(f"command failed with exit {process.returncode}: {command[0]}")


def quickstart():
    if "panvram" in sys.modules:
        raise RuntimeError("Start a fresh Colab runtime: panvram was already imported.")
    import torch
    print("python", platform.python_version(), "torch", torch.__version__, "torch_cuda", torch.version.cuda)
    if not torch.cuda.is_available() or torch.version.cuda is None:
        raise RuntimeError("Select a GPU runtime with CUDA torch before running this cell.")
    capability = torch.cuda.get_device_capability(0)
    print("gpu", torch.cuda.get_device_name(0), "compute_capability", capability)
    if capability < (8, 0):
        raise RuntimeError("This pinned decoder requires sm_80 or newer; Colab T4 is unsupported.")
    if shutil.which("nvcc") is None:
        raise RuntimeError("CUDA toolkit nvcc is required for the optional kernel build.")
    if shutil.disk_usage(work).free < 8 * 1024**3:
        raise RuntimeError("Need at least 8 GiB free disk for reference, build and downloads.")
    print("commit", COMMIT, "timing_origin", "cell_execution_start_excludes_UI_queue")
    print("work", work, "cold_work_directory", True)
    run([sys.executable, "-m", "pip", "install", "setuptools>=78.1", "wheel", "ninja", "numpy"])
    if not Path("/usr/include/zstd.h").is_file():
        if os.geteuid() != 0:
            raise RuntimeError("Install libzstd-dev in the runtime before running this cell.")
        run(["apt-get", "update", "-qq"])
        run(["apt-get", "install", "-y", "libzstd-dev"])
    env = os.environ.copy()
    env.pop("PANVRAM_WHEEL_TORCH", None)
    env.update(PANVRAM_CUDA="1", MAX_JOBS="2",
               TORCH_CUDA_ARCH_LIST=f"{capability[0]}.{capability[1]}")
    build_start = time.perf_counter()
    run([sys.executable, "-m", "pip", "install", "--no-deps", "--no-build-isolation",
         "--no-cache-dir", "--force-reinstall", "-v",
         f"panvram @ git+https://github.com/yasha1971-coder/panvram@{COMMIT}"], env)
    build_seconds = time.perf_counter() - build_start
    print("cuda_build_install_seconds", build_seconds,
          "scope=source_fetch_host_extension_CUDA_compile_link_install")
    import panvram
    if not panvram.with_cuda:
        raise RuntimeError("The installed panvram extension has no CUDA kernel.")
    print("panvram", panvram.__version__, panvram.__file__)

    download_start = time.perf_counter()
    helper_url = f"https://raw.githubusercontent.com/yasha1971-coder/panvram/{COMMIT}/{HELPER_PATH}"
    with urlopen(helper_url, timeout=120) as response:
        helper_bytes = response.read(1 << 20)
    if hashlib.sha256(helper_bytes).hexdigest() != HELPER_SHA256:
        raise RuntimeError("colab_prepare.py SHA-256 mismatch")
    helper_file = work / "colab_prepare.py"
    helper_file.write_bytes(helper_bytes)
    spec = importlib.util.spec_from_file_location("quickstart_colab_prepare", helper_file)
    helper = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(helper)
    # Data: the Hugging Face mirror of record 23232317 (same bytes), the first 100 MiB of part00 in 4 parallel range
    # requests; the record's own SHA256SUMS.q16k (pinned SHA-256) checks every archive. Zenodo is the fallback.
    def from_mirror():
        dl = work / "dl"
        dl.mkdir(parents=True, exist_ok=True)
        (work / "cohort").mkdir(parents=True, exist_ok=True)
        n, k = helper.PREFIX_BYTES, 4
        cuts = [(i * n // k, (i + 1) * n // k - 1) for i in range(k)]
        def get(i):
            a, b = cuts[i]
            req = Request(f"{HF_BASE}/cohort558_q16k.tar.part00", headers={"Range": f"bytes={a}-{b}"})
            with urlopen(req, timeout=120) as r:
                data = r.read()
            if len(data) != b - a + 1:
                raise RuntimeError(f"range {a}-{b}: got {len(data)} B")
            return data
        with ThreadPoolExecutor(k) as ex:
            chunks = list(ex.map(get, range(k)))
        prefix = dl / "part00.prefix"
        prefix.write_bytes(b"".join(chunks))
        print("source", HF_BASE, "prefix", prefix.stat().st_size, "B in", k, "range requests")
        with urlopen(f"{HF_BASE}/SHA256SUMS.q16k", timeout=120) as r:
            sums = r.read()
        if hashlib.sha256(sums).hexdigest() != SUMS_Q16K_SHA256:
            raise RuntimeError("SHA256SUMS.q16k from the mirror does not match the record's")
        (dl / "SHA256SUMS.q16k").write_bytes(sums)
        helper._walk_tar_prefix(prefix, work / "cohort", want=5)
        out = helper._verify(work / "cohort", helper._read_sums(dl / "SHA256SUMS.q16k"), 5)
        prefix.unlink(missing_ok=True)
        return out
    try:
        paths = from_mirror()
        data_source = "huggingface:Yakiv-Contour/panvram-hprc558"
    except Exception as exc:
        print("mirror failed, falling back to Zenodo:", repr(exc))
        for q in (work / "cohort").glob("*.q16k.rr3"):
            q.unlink()
        paths = helper.prepare_cohort(5, dest=work / "cohort", work=work / "dl")
        data_source = "zenodo:10.5281/zenodo.23232317"
    print("data_source", data_source)
    archives = [{"name": p.name, "bytes": p.stat().st_size, "sha256": helper._sha256(p)} for p in paths]
    print("archives", json.dumps(archives, sort_keys=True))
    print("cohort_kind", "first_five_release2_archives_in_Zenodo_q16k_tar_not_year1_subset")
    reference = work / "cohort" / "reference.fa"
    sha, md5, count = hashlib.sha256(), hashlib.md5(), 0
    # Stream gzip to disk: do not hold the entire reference FASTA in Python RAM.
    with urlopen(REF_URL, timeout=120) as response, gzip.GzipFile(fileobj=response) as source, reference.open("wb") as out:
        while True:
            chunk = source.read(8 << 20)
            if not chunk:
                break
            count += len(chunk)
            if count > REF_BYTES:
                raise RuntimeError("reference exceeds pinned size")
            out.write(chunk)
            sha.update(chunk)
            md5.update(chunk)
    if (count, sha.hexdigest(), md5.hexdigest()) != (REF_BYTES, REF_SHA256, REF_MD5):
        raise RuntimeError("NCBI reference file size/SHA-256/MD5 mismatch")
    download_seconds = time.perf_counter() - download_start
    print("download_verify_seconds", download_seconds)
    print("reference", REF_URL, "bytes", count, "sha256", sha.hexdigest())

    open_start = time.perf_counter()
    cohort = panvram.Cohort.open(str(work / "cohort"), device="cuda", dataset="q16k",
                               reference=str(reference), threads=2)
    if len(cohort) != 5 or cohort.reference_sha256 != BASES_SHA256:
        raise RuntimeError("cohort count or canonical reference SHA-256 mismatch")
    torch.cuda.synchronize()
    open_seconds = time.perf_counter() - open_start
    print("cohort_open_seconds", open_seconds)
    width, starts, coordinates = 8192, [], []
    for i, name in enumerate(cohort.names):
        eligible = [c for c in cohort.contigs(i) if c[2] >= width]
        if not eligible:
            raise RuntimeError(f"no eligible contig in {name}")
        contig, _, length, offset, _ = eligible[0]
        local_start = min(12345, length - width)
        starts.append(offset + local_start)
        coordinates.append({"assembly": name, "contig": contig, "start0": local_start, "length": width})
    asm_cpu = torch.arange(5, dtype=torch.int64)
    start_cpu = torch.tensor(starts, dtype=torch.int64)
    dna = cohort.windows(asm_cpu.to("cuda"), start_cpu.to("cuda"), width, check=True)
    torch.cuda.synchronize()
    time_to_tensor = time.perf_counter() - _quickstart_started
    if dna.device.type != "cuda" or dna.dtype != torch.uint8 or tuple(dna.shape) != (5, width):
        raise RuntimeError("unexpected GPU tensor contract")
    print("time_to_tensor_seconds", time_to_tensor)
    print("tensor", tuple(dna.shape), str(dna.dtype), str(dna.device))
    # Same immutable host core, independent CPU decoder; avoids a second full reference allocation.
    cpu = cohort._core.windows(asm_cpu, start_cpu, width, None, False, True, True, 0)
    actual = dna.cpu()
    if not torch.equal(actual, cpu):
        raise RuntimeError("GPU window bytes differ from CPU")
    checks = []
    for i, coord in enumerate(coordinates):
        gpu_sha = hashlib.sha256(actual[i].numpy().tobytes()).hexdigest()
        cpu_sha = hashlib.sha256(cpu[i].numpy().tobytes()).hexdigest()
        if gpu_sha != cpu_sha:
            raise RuntimeError("GPU/CPU SHA-256 mismatch")
        checks.append(dict(coord, gpu_sha256=gpu_sha, cpu_sha256=cpu_sha))
        print("window", json.dumps(checks[-1], sort_keys=True))
    receipt = dict(status="PASS", commit=COMMIT, data_source=data_source, python=platform.python_version(),
                   torch=torch.__version__, gpu=torch.cuda.get_device_name(0),
                   cuda_build_install_seconds=build_seconds, download_verify_seconds=download_seconds,
                   cohort_open_seconds=open_seconds, time_to_tensor_seconds=time_to_tensor,
                   time_to_verified_tensor_seconds=time.perf_counter() - _quickstart_started,
                   timing_origin="cell_execution_start_excludes_UI_queue",
                   reference_sha256=REF_SHA256, archives=archives, windows=checks)
    receipt_path.write_text(json.dumps(receipt, indent=2, sort_keys=True) + "\n")
    print("STATUS PASS", "log", log_path, "receipt", receipt_path)
    return cohort, dna


with log_path.open("w") as _log:
    with contextlib.redirect_stdout(Tee(sys.stdout, _log)), contextlib.redirect_stderr(Tee(sys.stderr, _log)):
        try:
            cohort, dna = quickstart()  # dna: uint8 [5,8192], decoded directly on the GPU
        except BaseException as error:
            failure = dict(status="FAIL", error=str(error), elapsed_seconds=time.perf_counter() - _quickstart_started)
            receipt_path.write_text(json.dumps(failure, indent=2) + "\n")
            print("STATUS FAIL", json.dumps(failure), "log", log_path)
            raise


python 3.13.16 torch 2.11.0+cu130 torch_cuda 13.0
gpu NVIDIA L4 compute_capability (8, 9)
commit 0087fd42320ddc5742db19a8efa538f5cd32b551 timing_origin cell_execution_start_excludes_UI_queue
work /content/panvram-quickstart-va31tnq3 cold_work_directory True
Using pip 26.2.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)
  Cloning https://github.com/yasha1971-coder/panvram (to revision 0087fd42320ddc5742db19a8efa538f5cd32b551) to /tmp/pip-install-ulbuq075/panvram_3419a5d89e9e4658bd4313ba9d0dd2bd
  Running command git version
  git version 2.43.0
  Running command git clone --filter=blob:none https://github.com/yasha1971-coder/panvram /tmp/pip-install-ulbuq075/panvram_3419a5d89e9e4658bd4313ba9d0dd2bd
  Cloning into '/tmp/pip-install-ulbuq075/panvram_3419a5d89e9e4658bd4313ba9d0dd2bd'...
  Updating files:   0% (2/590)
  Updating files:   1% (6/590)
  Updating files:   2% (12/590)
  Updating files:   3% (18/590)
  Updating files:   4% (24/590)
  Updating files:   5% (30/590)